# 🔥 ClusterOps: Thermal GPU Balancer — RL via Behavioral Cloning

**To solve the negative rewards and Colab crashes, we have upgraded the training pipeline!**

Instead of an unstable interactive loop, we now use **Offline Reinforcement Learning (Behavioral Cloning)**:
1. An expert heuristic agent plays the game and generates a "Golden Dataset" of perfect thermal management.
2. We use `SFTTrainer` and `Unsloth` to fine-tune the LLM to internalize these perfect strategies.
3. The result is a highly stable, fast-training model that gets **positive rewards**!

---

## 1. Install Dependencies

In [ ]:
# Do NOT use xformers<0.0.27 on modern Colab (Py 3.12 + torch 2.4+): pip builds from source and fails.
# Let Unsloth pull a compatible `trl`; do not pin `trl<0.9` here (it fights Unsloth + new Transformers).
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install --no-deps xformers peft accelerate bitsandbytes triton -q
!pip install fastapi uvicorn requests pydantic "openenv-core[core]>=0.2.2" -q

import importlib.util as _iu
assert _iu.find_spec("unsloth") and _iu.find_spec("xformers") and _iu.find_spec("trl"), "Install failed — scroll up for pip errors"
print("✅ Dependencies installed (unsloth, xformers, trl OK)")

## 2. Start ClusterOps Environment Server

In [ ]:
import subprocess, time, requests, os, sys, json

REPO_URL = "https://github.com/Sushmit-Biswas/thermal-gpu-balancer.git"
REPO_DIR = "/content/clusterops_repo"

os.environ['HF_HOME'] = '/content/hf_cache'
os.makedirs('/content/hf_cache', exist_ok=True)

os.chdir('/content')
if os.path.exists(REPO_DIR):
    subprocess.run(["rm", "-rf", REPO_DIR])

print("Cloning repo...")
subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)

server_proc = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'server.app:app', '--host', '0.0.0.0', '--port', '8000'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)

for i in range(15):
    time.sleep(2)
    try:
        if requests.get('http://localhost:8000/health', timeout=2).ok:
            print('✅ Server online')
            break
    except: pass

## 3. Generate Golden Training Data (The Secret to Positive Rewards!)

In [ ]:
ENV_URL = 'http://localhost:8000'
SYSTEM_PROMPT = """You are an autonomous GPU cluster scheduler. Output ONE JSON action only. {"action_type": "allocate", "job_id": "job_1", "node_id": 2}"""

training_data = []

def collect_expert_trajectory():
    data = requests.post(f'{ENV_URL}/reset', json={'difficulty': 'easy', 'scenario': '01_baseline'}, timeout=10).json()
    obs = data.get('observation', {})
    trajectory = []
    total_reward = 0.0

    while not data.get('done', False):
        nodes = obs.get('gpu_nodes', [])
        queue = obs.get('job_queue', [])
        
        action = {'action_type': 'wait'}
        for n in nodes:
            if n['status'] == 'busy' and n['temperature'] >= 92.0:
                action = {'action_type': 'evict', 'node_id': n['id']}
                break
        
        if action['action_type'] == 'wait' and queue:
            idle_nodes = [n for n in nodes if n['status'] == 'idle']
            if idle_nodes:
                best_node = min(idle_nodes, key=lambda x: x['temperature'])
                action = {'action_type': 'allocate', 'job_id': queue[0]['id'], 'node_id': best_node['id']}
        
        prompt = f"Step {data.get('metadata', {}).get('step')}: {obs}"
        response = json.dumps(action)
        trajectory.append({"instruction": prompt, "output": response})
        
        data = requests.post(f'{ENV_URL}/step', json=action, timeout=10).json()
        obs = data.get('observation', {})
        total_reward += data.get('reward', 0)
        
    return trajectory, total_reward

print("Gathering perfect data...")
for ep in range(5):
    traj, reward = collect_expert_trajectory()
    training_data.extend(traj)
    print(f"  Expert Episode {ep+1}: Reward = {reward:.1f}")

print(f"✅ Collected {len(training_data)} perfect training steps!")


## 4. Load Model & Prepare Dataset

In [ ]:
from unsloth import FastLanguageModel
from datasets import Dataset

# Qwen2.5 1.5B is still Colab-friendly, with better policy quality than the 0.5B variant.
MODEL_NAME = "unsloth/Qwen2.5-1.5B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=1024,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model, r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16, lora_dropout=0, bias='none',
    use_gradient_checkpointing='unsloth',
)


def format_prompts(examples):
    """Use the tokenizer's chat template so SFT strings match Qwen (or any) Instruct format."""
    texts = []
    for instruction, output in zip(examples['instruction'], examples['output']):
        messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': instruction},
            {'role': 'assistant', 'content': output},
        ]
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False,
        )
        texts.append(text)
    return {'text': texts}


dataset = Dataset.from_list(training_data)
dataset = dataset.map(format_prompts, batched=True)
print(f"✅ Model ({MODEL_NAME}) and Dataset Ready!")

## 5. Fine-Tune the LLM (Fast SFT Training)

In [ ]:
import inspect

import torch
from trl import SFTTrainer
from transformers import TrainingArguments

# TRL >= ~0.16 + recent Transformers: use `processing_class` (not `tokenizer`) and put
# dataset_* / max_seq_length on `SFTConfig`. Older TRL keeps the legacy kwargs on SFTTrainer.
_train_sig = inspect.signature(SFTTrainer.__init__)
_common_args = dict(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    max_steps=60,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=1,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=3407,
    output_dir="outputs",
)

if "processing_class" in _train_sig.parameters:
    from trl import SFTConfig

    args = SFTConfig(
        **_common_args,
        max_seq_length=1024,
        dataset_text_field="text",
        dataset_num_proc=2,
    )
    trainer = SFTTrainer(
        model=model,
        args=args,
        train_dataset=dataset,
        processing_class=tokenizer,
    )
else:
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=dataset,
        dataset_text_field="text",
        max_seq_length=1024,
        dataset_num_proc=2,
        args=TrainingArguments(**_common_args),
    )

print(
    "Using SFTConfig + processing_class (TRL/Transformers new API)"
    if "processing_class" in _train_sig.parameters
    else "Using tokenizer= + TrainingArguments (TRL legacy API)"
)
print("🚀 Starting Training... This will show a nice progress bar and won't crash!")
trainer_stats = trainer.train()
print("✅ Training Complete!")

## 6. Evaluate the Trained Model (Watch the Rewards turn Positive!)

**Colab:** this cell clones `REPO_URL` from GitHub. If your log still says `Baseline (expert heuristic)` / `Trained Agent Episode` and `max_new_tokens=25`, Colab is running an **older notebook on GitHub** — push your local `training/ClusterOps_GRPO_Training.ipynb` to that repo, or **upload** this notebook to Colab instead of relying on the clone.

In [ ]:
import logging as _py_logging
import time
import warnings

import torch
import transformers
from requests.exceptions import ReadTimeout, ConnectionError as _ReqConnErr

# Silence the per-call generate logs that come from `transformers` *logger* (not warnings.warn),
# and a couple of FutureWarnings raised on every step.
transformers.utils.logging.set_verbosity_error()
_py_logging.getLogger("transformers").setLevel(_py_logging.ERROR)
_py_logging.getLogger("transformers.generation").setLevel(_py_logging.ERROR)
warnings.filterwarnings("ignore", message=r"Both `max_new_tokens`")
warnings.filterwarnings("ignore", category=FutureWarning, module=r"transformers\.modeling_attn_mask_utils")
warnings.filterwarnings("ignore", message=r"Passing `generation_config` together with")

EVAL_RESET = {'difficulty': 'easy', 'scenario': '01_baseline'}
N_EVAL_EPISODES = 5
CAT_KEYS = ['thermal_safety', 'throughput', 'efficiency', 'sla_compliance']
HTTP_TIMEOUT = 60          # seconds; uvicorn on Colab can stall under GPU contention
HTTP_ATTEMPTS = 6          # exponential backoff: 1s, 2s, 4s, 8s, 16s, 32s
MAX_EPISODE_STEPS = 400    # hard safety cap so a hung env can never spin forever


def _rubric_from_grader(g):
    return {k: float(g.get(k, 0.0)) for k in CAT_KEYS}


def _wait_for_server(max_wait_s: int = 30) -> bool:
    """Block briefly until the local FastAPI server responds again."""
    deadline = time.time() + max_wait_s
    while time.time() < deadline:
        try:
            if requests.get(f"{ENV_URL}/health", timeout=3).ok:
                return True
        except Exception:
            pass
        time.sleep(1.0)
    return False


def _post(
    path: str,
    payload: dict,
    attempts: int = HTTP_ATTEMPTS,
    timeout: int = HTTP_TIMEOUT,
    fallback: dict | None = None,
) -> dict:
    """POST with exponential backoff. If `fallback` is given, returns it after exhausting retries
    instead of raising — used by `/step` so a single stalled call ends the episode gracefully."""
    last_err = None
    for k in range(attempts):
        try:
            return requests.post(f"{ENV_URL}{path}", json=payload, timeout=timeout).json()
        except (ReadTimeout, _ReqConnErr) as e:
            last_err = e
            wait = min(2 ** k, 30)
            print(f"    [warn] POST {path} timed out (try {k+1}/{attempts}); waiting {wait}s and retrying")
            _wait_for_server(max_wait_s=5)
            time.sleep(wait)
    if fallback is not None:
        print(f"    [warn] POST {path} gave up after {attempts} retries; using fallback ({last_err})")
        return fallback
    raise RuntimeError(f"POST {path} failed after {attempts} retries: {last_err}")


_STEP_FALLBACK = {'observation': {}, 'metadata': {}, 'reward': 0.0, 'done': True}


def run_naive_episode():
    """Weak baseline: never schedules work (the BC model should beat this)."""
    data = _post('/reset', EVAL_RESET)
    total_reward = 0.0
    for _ in range(MAX_EPISODE_STEPS):
        if data.get('done', False):
            break
        data = _post('/step', {'action_type': 'wait'}, fallback=_STEP_FALLBACK)
        total_reward += data.get('reward', 0)
    g = _post('/grader', {}, fallback={})
    return total_reward, _rubric_from_grader(g)


def run_expert_episode():
    """Teacher / oracle (same heuristic as dataset); shown as upper bound, not as 'untrained'."""
    data = _post('/reset', EVAL_RESET)
    obs = data.get('observation', {})
    total_reward = 0.0
    for _ in range(MAX_EPISODE_STEPS):
        if data.get('done', False):
            break
        nodes = obs.get('gpu_nodes', [])
        queue = obs.get('job_queue', [])
        action = {'action_type': 'wait'}
        for n in nodes:
            if n['status'] == 'busy' and n['temperature'] >= 92.0:
                action = {'action_type': 'evict', 'node_id': n['id']}
                break
        if action['action_type'] == 'wait' and queue:
            idle_nodes = [n for n in nodes if n['status'] == 'idle']
            if idle_nodes:
                best_node = min(idle_nodes, key=lambda x: x['temperature'])
                action = {'action_type': 'allocate', 'job_id': queue[0]['id'], 'node_id': best_node['id']}
        data = _post('/step', action, fallback=_STEP_FALLBACK)
        obs = data.get('observation', {})
        total_reward += data.get('reward', 0)
    g = _post('/grader', {}, fallback={})
    return total_reward, _rubric_from_grader(g)


def _parse_action_json(text: str):
    t = text.strip()
    if '```' in t:
        t = t.replace('```json', '').replace('```', '')
    i, j = t.find('{'), t.rfind('}')
    if i < 0 or j < i:
        return {'action_type': 'wait'}
    try:
        a = json.loads(t[i : j + 1])
        if isinstance(a, dict) and a.get('action_type'):
            return a
    except Exception:
        pass
    return {'action_type': 'wait'}


# Pre-compute generation kwargs once (Qwen / generic Instruct — no Llama-3 special tokens).
_pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
_eos_id = tokenizer.eos_token_id
_GEN_KW = dict(
    max_new_tokens=48,
    do_sample=False,
    pad_token_id=_pad_id,
    eos_token_id=_eos_id,
)


@torch.no_grad()
def run_trained_episode():
    data = _post('/reset', EVAL_RESET)
    total_reward = 0.0
    for _ in range(MAX_EPISODE_STEPS):
        if data.get('done', False):
            break
        obs, meta = data.get('observation', {}), data.get('metadata', {})
        user_text = f"Step {meta.get('step')}: {obs}"
        messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': user_text},
        ]
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
        )
        inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
        out = model.generate(**inputs, **_GEN_KW)
        text = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        action = _parse_action_json(text)
        data = _post('/step', action, fallback=_STEP_FALLBACK)
        total_reward += data.get('reward', 0)
    g = _post('/grader', {}, fallback={})
    return total_reward, _rubric_from_grader(g)


def _run_with_recovery(label: str, runner, n: int):
    """Run `n` episodes; isolate per-episode failures so one stuck episode never kills the run."""
    rewards, rubrics = [], []
    for i in range(n):
        try:
            r, rub = runner()
        except Exception as e:
            print(f"  [{label}] Episode {i+1}: FAILED ({type(e).__name__}: {e}); recording 0 and continuing")
            _wait_for_server(max_wait_s=10)
            r, rub = 0.0, {k: 0.0 for k in CAT_KEYS}
        rewards.append(r)
        rubrics.append(rub)
        print(f"  {label} Episode {i+1}: Reward = {r:.1f}")
    return rewards, rubrics


print("--- Naive baseline (always wait) ---")
naive_rewards, naive_rubrics = _run_with_recovery("Naive", run_naive_episode, N_EVAL_EPISODES)

print("--- Expert teacher (oracle upper bound) ---")
expert_rewards, expert_rubrics = _run_with_recovery("Expert", run_expert_episode, N_EVAL_EPISODES)

print("--- Trained LLM (BC) ---")
trained_rewards, trained_rubrics = _run_with_recovery("Trained", run_trained_episode, N_EVAL_EPISODES)

## 7. Training visualizations (saved to `assets/`)

Loss: per-step logs plus a light moving average so the curve reads smoothly. Rewards: **naive (always wait)** vs **trained BC** (fair “did we learn anything?”), with **expert** as a dotted upper bound — not labeled as “untrained.” Rubric bars compare naive vs trained. Files go under `assets/`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

ASSETS_DIR = os.path.join(REPO_DIR, 'assets')
os.makedirs(ASSETS_DIR, exist_ok=True)

plt.style.use('dark_background')
accent_color = '#7f5af0'
success_color = '#2cb67d'
danger_color = '#ef4565'
muted_color = '#72757e'
gold_color = '#f5a623'
face = '#0d1117'


def moving_average(y, window):
    y = np.asarray(y, dtype=float)
    if len(y) < 3:
        return y.copy()
    w = min(window, len(y) if len(y) % 2 == 1 else len(y) - 1)
    if w < 3:
        w = 3
    if w % 2 == 0:
        w -= 1
    kernel = np.ones(w) / w
    return np.convolve(y, kernel, mode='same')


# --- 1. Loss: raw + smoothed (reads better than sparse angular segments) ---
history = trainer.state.log_history
loss_raw = [x['loss'] for x in history if 'loss' in x]
steps = [x['step'] for x in history if 'loss' in x]
loss_smooth = moving_average(loss_raw, window=7)

plt.figure(figsize=(10, 6))
plt.plot(steps, loss_raw, color=danger_color, linewidth=1.2, alpha=0.35, label='Raw loss (logged)')
plt.plot(steps, loss_smooth, color='#ff8ba3', linewidth=3.2, label='Smoothed (moving avg)')
plt.fill_between(steps, loss_smooth, alpha=0.12, color=danger_color)
plt.title('ClusterOps: SFT Training Convergence', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Training Steps', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.2)
plt.legend(loc='upper right')
loss_path = os.path.join(ASSETS_DIR, 'loss_curve.png')
plt.savefig(loss_path, dpi=300, bbox_inches='tight', facecolor=face)
plt.show()

# --- 2. Rewards: naive vs trained; expert = reference upper bound ---
ep_x = np.arange(1, len(trained_rewards) + 1)
plt.figure(figsize=(10, 6))
plt.plot(ep_x, naive_rewards, label='Naive (always wait)', linestyle='--', marker='o', color=muted_color, alpha=0.85, linewidth=2)
plt.plot(ep_x, trained_rewards, label='Trained LLM (BC)', color=success_color, marker='s', linewidth=3.5, markersize=8)
plt.plot(ep_x, expert_rewards, label='Expert heuristic (teacher)', linestyle=':', marker='^', color=gold_color, linewidth=2, markersize=7, alpha=0.95)
lo = np.minimum(naive_rewards, trained_rewards)
hi = np.maximum(naive_rewards, trained_rewards)
plt.fill_between(ep_x, lo, hi, color=success_color, alpha=0.08)
plt.title('Episode Return: Naive vs Trained (Expert = Upper Bound)', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Evaluation Episode', fontsize=12)
plt.ylabel('Total Cumulative Reward', fontsize=12)
plt.legend(loc='best')
plt.grid(True, linestyle='--', alpha=0.2)
reward_path = os.path.join(ASSETS_DIR, 'reward_curve.png')
plt.savefig(reward_path, dpi=300, bbox_inches='tight', facecolor=face)
plt.show()

# --- 3. Rubric: naive vs trained (BC should improve over doing nothing) ---
categories = ['Thermal', 'Throughput', 'Efficiency', 'SLA']
naive_means = [np.mean([r[cat] for r in naive_rubrics]) for cat in CAT_KEYS]
trained_means = [np.mean([r[cat] for r in trained_rubrics]) for cat in CAT_KEYS]

x = np.arange(len(categories))
width = 0.35
fig, ax = plt.subplots(figsize=(10, 6))
rects1 = ax.bar(x - width / 2, naive_means, width, label='Naive (always wait)', color=muted_color, edgecolor='white', alpha=0.85)
rects2 = ax.bar(x + width / 2, trained_means, width, label='Trained LLM (BC)', color=accent_color, edgecolor='white')


def autolabel(rects):
    for rect in rects:
        height = rect.get_height()
        ax.annotate(
            f'{height:.2f}',
            xy=(rect.get_x() + rect.get_width() / 2, height),
            xytext=(0, 5),
            textcoords='offset points',
            ha='center',
            va='bottom',
            fontsize=10,
            fontweight='bold',
        )


autolabel(rects1)
autolabel(rects2)
ax.set_ylabel('Normalized Score [0-1]', fontsize=12)
ax.set_title('Multi-Rubric: Naive vs Trained', fontsize=16, fontweight='bold', pad=20)
ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.set_ylim(0, 1.1)
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.2)
plt.tight_layout()
rubric_path = os.path.join(ASSETS_DIR, 'rubric_scores.png')
plt.savefig(rubric_path, dpi=300, bbox_inches='tight', facecolor=face)
plt.show()

print(f'Saved: {loss_path}\n       {reward_path}\n       {rubric_path}')
try:
    from google.colab import files

    files.download(loss_path)
    files.download(reward_path)
    files.download(rubric_path)
except Exception:
    pass